# NER with Ollama + Quantized Model

Replicating core NER experiments from `nlp-hw-1.ipynb` using a local quantized model via **Ollama** instead of the cloud-hosted Llama 3.2.

**Model**: `qwen2.5:7b-instruct-q4_K_M`  
Why: Strong multilingual performance (including Ukrainian), 7B parameters — good accuracy/speed tradeoff locally. Significantly larger than `llama3.2:3b` used in hw-1.

**Experiments replicated**:
1. Zero-shot NER
2. Few-shot NER (5 examples)
3. Evaluation: Precision / Recall / F1

In [4]:
!pip install ollama pandas scikit-learn -q

## 1. Model Setup

Make sure Ollama is running: `ollama serve`

Pull the model once (≈4.7 GB for q4_K_M quantization):
```bash
ollama pull qwen2.5:7b-instruct-q4_K_M
```

Alternative models if you want to compare:
- `mistral-nemo:12b-instruct-q4_K_M` — larger, better European language support
- `gemma2:9b-instruct-q4_K_M` — Google, decent multilingual
- `llama3.1:8b-instruct-q4_K_M` — upgrade over llama3.2:3b used in hw-1

In [5]:
import ollama

MODEL = "qwen2.5:7b-instruct-q4_K_M"

# Verify Ollama is running and model is available
available = [m.model for m in ollama.list().models]
print("Available models:", available)

if MODEL not in available:
    print(f"Pulling {MODEL} ...")
    ollama.pull(MODEL)
    print("Done.")
else:
    print(f"✓ {MODEL} ready")

Available models: ['qwen2.5:7b-instruct-q4_K_M', 'llama3.2:latest']
✓ qwen2.5:7b-instruct-q4_K_M ready


## 2. Data & Utilities

In [6]:
import json
import re
import sys
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

df = pd.read_csv(ROOT / "data" / "train.csv")
print(f"Dataset size: {len(df)}")
print(df.head(2))

# Same split as hw-1: 5 samples as few-shot examples, rest for evaluation
train_df, few_shot_df = train_test_split(df, test_size=5, random_state=42)
print(f"\nFew-shot pool: {len(few_shot_df)}, Eval pool: {len(train_df)}")

Dataset size: 391
                                                  id  \
0  582dab45ea473d07323e3c6a0415425c037e63b5a7ab0a...   
1  df0933559918588668cb4219c80908b33cdef041dce468...   

                                                text  \
0  Зрозуміло , що український бізнес почав викори...   
1  \nДолучилися до заходу заступник голови райдер...   

                                            entities  
0  [{"label": "MISC", "text": "\u041a\u0421\u0412...  
1  [{"label": "JOB", "text": "\u0437\u0430\u0441\...  

Few-shot pool: 5, Eval pool: 386


In [8]:
import re

from utils.decode import decode_entities
from utils.normalize_entities import normalize_entities
from evaluation.ner_f1_score import ner_f1_score, evaluate_dataset


def extract_json_from_output(raw: str) -> list[dict]:
    """Extract JSON array from model response (handles markdown fences and surrounding text)."""
    raw = re.sub(r"```(?:json)?", "", raw).strip()
    match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return []


def run_ner(text: str, prompt_template: str, model: str = MODEL) -> list[dict]:
    """Send text to Ollama, return parsed entities."""
    prompt = prompt_template.replace("{{TEXT_HERE}}", text)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return extract_json_from_output(response.message.content)


def load_prompt(filename: str) -> str:
    return (ROOT / "prompts" / filename).read_text(encoding="utf-8")


def parse_ground_truth(entities_str: str) -> list[dict]:
    return decode_entities(entities_str)


print("Utilities loaded.")

Utilities loaded.


## 3. Zero-Shot NER

Uses `ner-prompt.txt` — the same baseline prompt as in hw-1, no examples provided.

In [9]:
zero_shot_prompt = load_prompt("ner-prompt.txt")
print("=== Zero-shot prompt preview ===")
print(zero_shot_prompt[:500])

=== Zero-shot prompt preview ===
Ти — система розпізнавання іменованих сутностей (NER) для української мови. 
Твоє завдання — знайти всі сутності у тексті та повернути їх у форматі JSON.

Класи, які ти повинен знаходити (усього 13):
- ART	artifact, предмети, створені людиною
- DATE	date, календарні дати
- DOC	document, закони, угоди, паспорти, інші документи
- JOB	job title, професії та посади
- LOC	location, географічні назви
- MON	money, суми грошей
- ORG	organization, компанії, установи, заклади
- PCT	percentage, відсотки
- 


In [10]:
EVAL_SAMPLE = train_df.sample(10, random_state=42).reset_index(drop=True)

zero_shot_results = []
for _, row in EVAL_SAMPLE.iterrows():
    predicted = run_ner(row["text"], zero_shot_prompt)
    ground_truth = parse_ground_truth(row["entities"])
    metrics = ner_f1_score(predicted, ground_truth)
    zero_shot_results.append({
        "id": row["id"],
        "predicted": predicted,
        "ground_truth": ground_truth,
        **metrics,
    })
    print(f"[{row['id']}] P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1']:.3f}")

zs_df = pd.DataFrame(zero_shot_results)
print(f"\nZero-shot avg — P: {zs_df['precision'].mean():.3f}  R: {zs_df['recall'].mean():.3f}  F1: {zs_df['f1'].mean():.3f}")

[4895884bda6905c72612cfa205d20379078ea6c334fcc87e92be942da7fb5779] P=0.690 R=0.408 F1=0.513
[3c5a6ef07b54c7f6bf3da1077ce57398ca61e6d518a3e756685c80d33548fb48] P=0.308 R=0.143 F1=0.195
[c125b29305bbcabf0741705c71b277c53cf7a206c83a9c8f980bed36db920218] P=0.118 R=0.053 F1=0.073
[52091b87423bc52c3f392cc1eea95d3260cff2ce36d431020905c7ee22472f77] P=0.500 R=0.189 F1=0.275
[fa4c22b2b1fd96e64d1b039c17b8a1d1e7f126b4f2391216a13778f447df339d] P=0.500 R=0.214 F1=0.300
[d6bdd4b4bca6d927f128b22916803eb19bd858dde0e1688dc51c34d227e5af7b] P=0.462 R=0.167 F1=0.245
[7575d90829ac922da742bd7d991ae8ef0e703bcbf2fe4ca23716b7ffc82d313a] P=0.000 R=0.000 F1=0.000
[dc65cce65230948255c07759aa7c6e243c4a57ebec9f740be5af7546d28224d0] P=0.353 R=0.162 F1=0.222
[170edff19c196e13a0ff6489a31dfcd09d64481c18a0a278dce590ee486fdb9b] P=0.429 R=0.273 F1=0.333
[80af7ad7cb970595551adac2f28c60a84a32177a897742be968aecfae423c07a] P=0.667 R=0.235 F1=0.348

Zero-shot avg — P: 0.402  R: 0.184  F1: 0.250


In [13]:
for i in zero_shot_results:
    print("predicted:", i["predicted"])

predicted: [{'label': 'PERS', 'text': 'Олексія Вострікова'}, {'label': 'QUANT', 'text': '31-річного'}, {'label': 'TIME', 'text': 'грудні 2019 року'}, {'label': 'LOC', 'text': 'Луганську'}, {'label': 'LOC', 'text': 'Києва'}, {'label': 'ORG', 'text': 'ДП «Завод «Радіореле»'}, {'label': 'JOB', 'text': 'заступником директора'}, {'label': 'ORG', 'text': 'ДП «Спецагролізінг»'}, {'label': 'MON', 'text': '100 тис грн'}, {'label': 'MON', 'text': '2 млн гривень'}, {'label': 'QUANT', 'text': '160 тис грн'}, {'label': 'ORG', 'text': 'НАК «Нафтогаз України»'}, {'label': 'ORG', 'text': 'ДП «Укравтогаз»'}, {'label': 'ORG', 'text': 'ДП «Національна енергетична компанія «Укренерго»'}, {'label': 'ORG', 'text': "ДП «Державний експертний центр Міністерства охорони здоров'я України»"}, {'label': 'ORG', 'text': 'ТОВ «Укрфармгруп»'}, {'label': 'MON', 'text': '144 тис грн'}, {'label': 'ORG', 'text': 'АМКУ'}, {'label': 'JOB', 'text': '_спіймав її на тендерній змові_'}, {'label': 'ART', 'text': 'двох квартир у 

## 4. Few-Shot NER

Uses `ner-prompt-few-shot.txt` with 5 real examples from the training set — same setup as hw-1.

In [14]:
few_shot_template = load_prompt("ner-prompt-few-shot.txt")
example_template = load_prompt("few-shot-temp.txt")

# Build few-shot examples block from the 5 reserved samples (same as hw-1)
examples_block = ""
for idx, (_, row) in enumerate(few_shot_df.iterrows(), start=1):
    gt = parse_ground_truth(row["entities"])
    example = (
        example_template
        .replace("{{INDEX_HERE}}", str(idx))
        .replace("{{TEXT_HERE}}", row["text"])
        .replace("{{ENTITIES_HERE}}", json.dumps(gt, ensure_ascii=False))
    )
    examples_block += example + "\n"

few_shot_prompt = few_shot_template.replace("{{FEW_SHOT_EXAMPLES_HERE}}", examples_block)
print("=== Few-shot prompt preview (first 800 chars) ===")
print(few_shot_prompt[:800])

=== Few-shot prompt preview (first 800 chars) ===
Ти — система розпізнавання іменованих сутностей (NER) для української мови. 
Твоє завдання — знайти всі сутності у тексті та повернути їх у форматі JSON.

Класи, які ти повинен знаходити (усього 13):
- ART	artifact, предмети, створені людиною
- DATE	date, календарні дати
- DOC	document, закони, угоди, паспорти, інші документи
- JOB	job title, професії та посади
- LOC	location, географічні назви
- MON	money, суми грошей
- ORG	organization, компанії, установи, заклади
- PCT	percentage, відсотки
- PERIOD	period, часові інтервали
- PERS	person name, ім’я або прізвище людини
- QUANT	quantity, числові значення з одиницями
- TIME	time, час
- MISC	miscellaneous, інші сутності, які не підпадають під попередні категорії

❗ Формат відповіді:
Поверни JSON масив. Кожен елемент:
{
  "label": "<клас>",
 


In [15]:
few_shot_results = []
for _, row in EVAL_SAMPLE.iterrows():
    predicted = run_ner(row["text"], few_shot_prompt)
    ground_truth = parse_ground_truth(row["entities"])
    metrics = ner_f1_score(predicted, ground_truth)
    few_shot_results.append({
        "id": row["id"],
        "predicted": predicted,
        "ground_truth": ground_truth,
        **metrics,
    })
    print(f"[{row['id']}] P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1']:.3f}")

fs_df = pd.DataFrame(few_shot_results)
print(f"\nFew-shot avg — P: {fs_df['precision'].mean():.3f}  R: {fs_df['recall'].mean():.3f}  F1: {fs_df['f1'].mean():.3f}")

[4895884bda6905c72612cfa205d20379078ea6c334fcc87e92be942da7fb5779] P=0.677 R=0.429 F1=0.525
[3c5a6ef07b54c7f6bf3da1077ce57398ca61e6d518a3e756685c80d33548fb48] P=0.706 R=0.429 F1=0.533
[c125b29305bbcabf0741705c71b277c53cf7a206c83a9c8f980bed36db920218] P=0.267 R=0.211 F1=0.235
[52091b87423bc52c3f392cc1eea95d3260cff2ce36d431020905c7ee22472f77] P=0.500 R=0.270 F1=0.351
[fa4c22b2b1fd96e64d1b039c17b8a1d1e7f126b4f2391216a13778f447df339d] P=0.688 R=0.393 F1=0.500
[d6bdd4b4bca6d927f128b22916803eb19bd858dde0e1688dc51c34d227e5af7b] P=0.400 R=0.167 F1=0.235
[7575d90829ac922da742bd7d991ae8ef0e703bcbf2fe4ca23716b7ffc82d313a] P=0.000 R=0.000 F1=0.000
[dc65cce65230948255c07759aa7c6e243c4a57ebec9f740be5af7546d28224d0] P=0.588 R=0.270 F1=0.370
[170edff19c196e13a0ff6489a31dfcd09d64481c18a0a278dce590ee486fdb9b] P=0.458 R=0.333 F1=0.386
[80af7ad7cb970595551adac2f28c60a84a32177a897742be968aecfae423c07a] P=0.316 R=0.353 F1=0.333

Few-shot avg — P: 0.460  R: 0.285  F1: 0.347


In [16]:
for i in few_shot_results:
    print("predicted:", i["predicted"])

predicted: [{'label': 'PERS', 'text': 'Олексія Вострікова'}, {'label': 'DATE', 'text': '31-річного'}, {'label': 'JOB', 'text': 'керувати Рівненською службою автомобільних доріг'}, {'label': 'DATE', 'text': 'грядні 2019 року'}, {'label': 'LOC', 'text': 'Луганську'}, {'label': 'LOC', 'text': 'Києва'}, {'label': 'JOB', 'text': '_заступником директора_'}, {'label': 'ORG', 'text': 'ДП «Завод «Радіореле»'}, {'label': 'JOB', 'text': '_в. о. заступника директора_'}, {'label': 'ORG', 'text': 'ДП «Спецагролізінг»'}, {'label': 'MON', 'text': '100 тис грн.'}, {'label': 'DATE', 'text': '2017 році'}, {'label': 'ART', 'text': 'Audi Q7'}, {'label': 'MON', 'text': '2 млн гривень'}, {'label': 'PERS', 'text': 'Олексій Востріков'}, {'label': 'DATE', 'text': '2014 році'}, {'label': 'QUANT', 'text': 'двох паркомісць'}, {'label': 'MON', 'text': '160 тис грн.'}, {'label': 'PERS', 'text': 'Микола Востріков'}, {'label': 'ORG', 'text': 'НАК «Нафтогаз України»'}, {'label': 'ORG', 'text': 'ДП «Укравтогаз»'}, {'lab

## 5. Results Comparison

Zero-shot vs Few-shot for `qwen2.5:7b` vs hw-1 baseline (`llama3.2:3b`).

hw-1 zero-shot baseline: P=0.071, R=0.036, F1=0.048

In [17]:
summary = pd.DataFrame([
    {
        "experiment": f"hw-1 zero-shot (llama3.2:3b)",
        "precision": 0.0714,
        "recall": 0.0357,
        "f1": 0.0476,
    },
    {
        "experiment": f"zero-shot ({MODEL})",
        "precision": zs_df["precision"].mean(),
        "recall": zs_df["recall"].mean(),
        "f1": zs_df["f1"].mean(),
    },
    {
        "experiment": f"few-shot 5ex ({MODEL})",
        "precision": fs_df["precision"].mean(),
        "recall": fs_df["recall"].mean(),
        "f1": fs_df["f1"].mean(),
    },
])

summary = summary.round(4)
print(summary.to_string(index=False))

                               experiment  precision  recall     f1
             hw-1 zero-shot (llama3.2:3b)     0.0714  0.0357 0.0476
   zero-shot (qwen2.5:7b-instruct-q4_K_M)     0.4025  0.1844 0.2503
few-shot 5ex (qwen2.5:7b-instruct-q4_K_M)     0.4600  0.2854 0.3469
